# 05A — Sentiment Dataset Creation for Entity-Level AI Impact Analysis

Pipeline stage:

- load cleaned entity contexts and entity summary from Stage 04
- construct a stratified sentiment labeling pool
- optionally enrich the pool with lightweight metadata features
- run resumable LLM labeling for entity-conditioned sentiment
- build a cleaned training dataset for downstream sentiment modeling
- export diagnostics, summary tables, and run config

Input directory

`output/04_entity_extraction/04B_analysis_entities\`

- `entity_analysis_mentions.parquet`
- `entity_analysis_summary.parquet`
- `entity_contexts_final.parquet`

Output directory

`output/05_sentiment_dataset/`

- `sentiment_label_pool.parquet`
- `sentiment_llm_labels.jsonl`
- `sentiment_llm_labels.parquet`
- `sentiment_training_data.parquet`
- `sentiment_label_summary.csv`
- `sentiment_labeling_report.json`
- `sentiment_run_config.json`

In [1]:
!pip -q install pyarrow aiohttp tqdm

## Environment Setup

In [2]:
import os
import re
import gc
import json
import math
import time
import asyncio
import datetime as dt
from typing import Any, Dict, List, Optional

import aiohttp
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from tqdm.auto import tqdm

from google.colab import drive
drive.mount("/content/drive")

try:
    from google.colab import userdata
except Exception:
    userdata = None

Mounted at /content/drive


## Paths

In [3]:
BASE_DIR = "/content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen"

IN_DIR_04 = f"{BASE_DIR}/output/04_entity_extraction/04B_analysis_entities"
OUT_DIR_05 = f"{BASE_DIR}/output/05_sentiment_dataset"

ENTITY_ANALYSIS_MENTIONS_PARQUET = f"{IN_DIR_04}/entity_analysis_mentions.parquet"
ENTITY_ANALYSIS_SUMMARY_PARQUET = f"{IN_DIR_04}/entity_analysis_summary.parquet"
ENTITY_CONTEXTS_FINAL_PARQUET = f"{IN_DIR_04}/entity_contexts_final.parquet"

SENTIMENT_LABEL_POOL_PARQUET = f"{OUT_DIR_05}/sentiment_label_pool.parquet"
SENTIMENT_LLM_LABELS_JSONL = f"{OUT_DIR_05}/sentiment_llm_labels.jsonl"
SENTIMENT_LLM_LABELS_PARQUET = f"{OUT_DIR_05}/sentiment_llm_labels.parquet"
SENTIMENT_TRAINING_DATA_PARQUET = f"{OUT_DIR_05}/sentiment_training_data.parquet"
SENTIMENT_LABEL_SUMMARY_CSV = f"{OUT_DIR_05}/sentiment_label_summary.csv"
SENTIMENT_LABELING_REPORT_JSON = f"{OUT_DIR_05}/sentiment_labeling_report.json"
SENTIMENT_RUN_CONFIG_JSON = f"{OUT_DIR_05}/sentiment_run_config.json"

os.makedirs(OUT_DIR_05, exist_ok=True)

assert os.path.exists(ENTITY_ANALYSIS_MENTIONS_PARQUET), f"Missing: {ENTITY_ANALYSIS_MENTIONS_PARQUET}"
assert os.path.exists(ENTITY_ANALYSIS_SUMMARY_PARQUET), f"Missing: {ENTITY_ANALYSIS_SUMMARY_PARQUET}"
assert os.path.exists(ENTITY_CONTEXTS_FINAL_PARQUET), f"Missing: {ENTITY_CONTEXTS_FINAL_PARQUET}"

print("ENTITY_ANALYSIS_MENTIONS_PARQUET:", ENTITY_ANALYSIS_MENTIONS_PARQUET)
print("ENTITY_ANALYSIS_SUMMARY_PARQUET:", ENTITY_ANALYSIS_SUMMARY_PARQUET)
print("ENTITY_CONTEXTS_FINAL_PARQUET:", ENTITY_CONTEXTS_FINAL_PARQUET)
print("OUT_DIR_05:", OUT_DIR_05)

ENTITY_ANALYSIS_MENTIONS_PARQUET: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/04_entity_extraction/04B_analysis_entities/entity_analysis_mentions.parquet
ENTITY_ANALYSIS_SUMMARY_PARQUET: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/04_entity_extraction/04B_analysis_entities/entity_analysis_summary.parquet
ENTITY_CONTEXTS_FINAL_PARQUET: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/04_entity_extraction/04B_analysis_entities/entity_contexts_final.parquet
OUT_DIR_05: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_dataset


## Runtime and Configuration

In [4]:
SEED = 42
rng = np.random.default_rng(SEED)

REUSE_LABEL_POOL_IF_PRESENT = True
REUSE_LLM_LABELS_IF_PRESENT = False

TARGET_TOTAL_SAMPLES = 9000

TARGET_SAMPLES_BY_TYPE = {
    "company": 3200,
    "technology": 3200,
    "government_institution": 1400,
    "person": 1200,
}

ANALYSIS_KEEP_TYPES = set(TARGET_SAMPLES_BY_TYPE.keys())

MAX_CONTEXT_CHARS = 1400
MIN_CONTEXT_CHARS = 80

HEAD_DOCS_MIN = 300
MID_DOCS_MIN = 80
TAIL_DOCS_MIN = 20

HEAD_SHARE = 0.45
MID_SHARE = 0.35
TAIL_SHARE = 0.20

MAX_PER_ENTITY_HEAD = {
    "company": 18,
    "technology": 18,
    "government_institution": 10,
    "person": 8,
}
MAX_PER_ENTITY_MID = {
    "company": 10,
    "technology": 10,
    "government_institution": 6,
    "person": 5,
}
MAX_PER_ENTITY_TAIL = {
    "company": 5,
    "technology": 5,
    "government_institution": 4,
    "person": 4,
}

MIN_ENTITY_DOCS = {
    "company": 10,
    "technology": 10,
    "government_institution": 8,
    "person": 8,
}

MIN_ENTITY_ROWS = {
    "company": 15,
    "technology": 15,
    "government_institution": 10,
    "person": 10,
}

LOW_VALUE_PERSON_TERMS = {
    "he", "she", "they", "we", "i", "you", "users", "researchers", "scientists",
    "developers", "engineers", "students", "workers", "people", "consumers",
    "customers", "patients", "teachers", "parents", "children", "officials",
    "executives", "lawmakers", "analysts", "experts", "critics", "authors",
    "journalists", "publishers", "creators", "employees", "employers",
    "founders", "investors"
}

LOW_VALUE_GOV_TERMS = {
    "world", "europe", "asia", "africa", "america", "americas", "global", "international"
}

LOW_VALUE_TECH_TERMS = {
    "ai", "artificial intelligence", "ai models", "ai systems", "ai tools", "ai technology"
}

if userdata is not None:
    try:
        DEEPSEEK_API_KEY = userdata.get("DEEPSEEK_API_KEY").strip()
    except Exception:
        DEEPSEEK_API_KEY = ""
else:
    DEEPSEEK_API_KEY = os.environ.get("DEEPSEEK_API_KEY", "").strip()

USE_LLM_LABELING = bool(DEEPSEEK_API_KEY)

DEEPSEEK_BASE_URL = "https://api.deepseek.com"
DEEPSEEK_ENDPOINT = f"{DEEPSEEK_BASE_URL}/chat/completions"
DEEPSEEK_MODEL = "deepseek-chat"

LLM_CONCURRENCY = 48
LLM_MAX_RETRIES = 5
LLM_TIMEOUT_S = 90
CHECKPOINT_EVERY = 200
JSONL_FLUSH_EVERY = 100

VALID_SENTIMENT_LABELS = {"positive", "negative", "mixed_or_unclear"}

run_config = {
    "seed": SEED,
    "reuse_label_pool_if_present": REUSE_LABEL_POOL_IF_PRESENT,
    "reuse_llm_labels_if_present": REUSE_LLM_LABELS_IF_PRESENT,
    "target_total_samples": TARGET_TOTAL_SAMPLES,
    "target_samples_by_type": TARGET_SAMPLES_BY_TYPE,
    "analysis_keep_types": sorted(ANALYSIS_KEEP_TYPES),
    "max_context_chars": MAX_CONTEXT_CHARS,
    "min_context_chars": MIN_CONTEXT_CHARS,
    "head_docs_min": HEAD_DOCS_MIN,
    "mid_docs_min": MID_DOCS_MIN,
    "tail_docs_min": TAIL_DOCS_MIN,
    "head_share": HEAD_SHARE,
    "mid_share": MID_SHARE,
    "tail_share": TAIL_SHARE,
    "max_per_entity_head": MAX_PER_ENTITY_HEAD,
    "max_per_entity_mid": MAX_PER_ENTITY_MID,
    "max_per_entity_tail": MAX_PER_ENTITY_TAIL,
    "min_entity_docs": MIN_ENTITY_DOCS,
    "min_entity_rows": MIN_ENTITY_ROWS,
    "use_llm_labeling": USE_LLM_LABELING,
    "llm_model": DEEPSEEK_MODEL if USE_LLM_LABELING else None,
    "llm_concurrency": LLM_CONCURRENCY,
    "llm_max_retries": LLM_MAX_RETRIES,
    "llm_timeout_s": LLM_TIMEOUT_S,
    "checkpoint_every": CHECKPOINT_EVERY,
    "jsonl_flush_every": JSONL_FLUSH_EVERY,
}

with open(SENTIMENT_RUN_CONFIG_JSON, "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2)

print("USE_LLM_LABELING:", USE_LLM_LABELING)
print("wrote:", SENTIMENT_RUN_CONFIG_JSON)

USE_LLM_LABELING: True
wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_dataset/sentiment_run_config.json


## Utility Functions

In [5]:
WS_RE = re.compile(r"\s+")

def normalize_spaces(text: str) -> str:
    return WS_RE.sub(" ", str(text or "")).strip()

def normalize_entity_key(text: str) -> str:
    x = normalize_spaces(str(text or "")).lower()
    x = re.sub(r"['’]", "", x)
    x = re.sub(r"[\(\)\[\]\{\}\.,;:!?\"“”‘’/\\|]+", " ", x)
    x = re.sub(r"[-_]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def to_jsonable(obj: Any):
    if obj is None:
        return None
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    if isinstance(obj, (dt.date, dt.datetime, pd.Timestamp)):
        return obj.isoformat()
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(x) for x in obj]
    if isinstance(obj, float):
        if math.isnan(obj) or math.isinf(obj):
            return None
        return obj
    return obj

def build_time_bucket(series: pd.Series) -> pd.Series:
    dt_series = pd.to_datetime(series, errors="coerce")
    return dt_series.dt.to_period("Q").astype(str)

def compact_text(text: str, max_chars: int) -> str:
    s = normalize_spaces(text)
    if len(s) <= max_chars:
        return s
    return s[:max_chars].rstrip() + " ..."

def sample_with_entity_cap(
    df: pd.DataFrame,
    target_n: int,
    entity_col: str,
    max_per_entity: int,
    seed: int = SEED,
) -> pd.DataFrame:
    if len(df) == 0 or target_n <= 0:
        return df.head(0).copy()

    parts = []
    for _, sub in df.groupby(entity_col, dropna=False):
        take_n = min(max_per_entity, len(sub))
        parts.append(sub.sample(n=take_n, random_state=seed))

    capped = pd.concat(parts, ignore_index=True) if parts else df.head(0).copy()

    if len(capped) <= target_n:
        return capped.copy()

    return capped.sample(n=target_n, random_state=seed).copy()

def load_done_ids(path: str, id_col: str) -> set:
    if not os.path.exists(path):
        return set()
    done = set()
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
                if id_col in obj:
                    done.add(obj[id_col])
            except Exception:
                continue
    return done

def is_low_value_entity(entity: str, final_type: str) -> bool:
    key = normalize_entity_key(entity)
    if final_type == "person" and key in LOW_VALUE_PERSON_TERMS:
        return True
    if final_type == "government_institution" and key in LOW_VALUE_GOV_TERMS:
        return True
    if final_type == "technology" and key in LOW_VALUE_TECH_TERMS:
        return True
    return False

def schema_check(df: pd.DataFrame, required_cols: List[str], name: str) -> None:
    missing = [c for c in required_cols if c not in df.columns]
    assert not missing, f"{name} missing required columns: {missing}"

## Load Stage 04 Outputs

In [6]:
entity_analysis_mentions = pd.read_parquet(ENTITY_ANALYSIS_MENTIONS_PARQUET)
entity_analysis_summary = pd.read_parquet(ENTITY_ANALYSIS_SUMMARY_PARQUET)
entity_contexts_final = pd.read_parquet(ENTITY_CONTEXTS_FINAL_PARQUET)

schema_check(
    entity_analysis_mentions,
    [
        "doc_id", "block_id", "block_uid", "url", "date", "title", "domain",
        "clean_block_text", "clean_block_len", "canonical_entity", "final_type", "confidence"
    ],
    "entity_analysis_mentions"
)

schema_check(
    entity_analysis_summary,
    ["canonical_entity", "final_type", "n_docs", "n_rows", "n_domains", "confidence_mean"],
    "entity_analysis_summary"
)

schema_check(
    entity_contexts_final,
    ["canonical_entity", "final_type", "domain", "title", "date", "clean_block_text", "confidence", "doc_id", "block_id", "url"],
    "entity_contexts_final"
)

print("entity_analysis_mentions shape:", entity_analysis_mentions.shape)
print("entity_analysis_summary shape:", entity_analysis_summary.shape)
print("entity_contexts_final shape:", entity_contexts_final.shape)

display(entity_analysis_mentions.head(5))
display(entity_analysis_summary.head(5))
display(entity_contexts_final.head(5))

entity_analysis_mentions shape: (2694302, 26)
entity_analysis_summary shape: (287492, 7)
entity_contexts_final shape: (614017, 11)


,doc_id,block_id,block_uid,url,date,language,title,domain,clean_block_text,p_ai_block,...,char_start,char_end,final_type_raw,confidence,block_text_hash,is_news_source,is_numeric_junk,canonical_entity,final_type,canonical_key
0,0,0,0:0,https://blockworks.co/price/bad,2025-06-23,en,"Bad Idea AI Price (BAD), Market Cap, Price Tod...",blockworks.co,The price of BAD is down -0.61% since last hou...,0.990286,...,1035,1048,person,0.903304,a75c4f305d9d5292bf75dda182b3ce1e1043f7cf,False,False,Byron Gilliam,person,byron gilliam
1,0,0,0:0,https://blockworks.co/price/bad,2025-06-23,en,"Bad Idea AI Price (BAD), Market Cap, Price Tod...",blockworks.co,The price of BAD is down -0.61% since last hou...,0.990286,...,1447,1468,person,0.646446,a75c4f305d9d5292bf75dda182b3ce1e1043f7cf,False,False,Carlos Gonzalez Campo,person,carlos gonzalez campo
2,2,6,2:6,https://boingboing.net/2024/09/18/if-using-ai-...,2024-09-22,en,"If using AI feels like a chore, try this - Boi...",boingboing.net,AI isn't going anywhere. The problem is that w...,0.998094,...,481,486,product,0.770057,3b8a48a79501909cb5de04b51b5e37be6b07f931,False,False,GPT-4,technology,gpt 4
3,2,6,2:6,https://boingboing.net/2024/09/18/if-using-ai-...,2024-09-22,en,"If using AI feels like a chore, try this - Boi...",boingboing.net,AI isn't going anywhere. The problem is that w...,0.998094,...,498,508,product,0.906617,3b8a48a79501909cb5de04b51b5e37be6b07f931,False,False,Gemini Pro,technology,gemini pro
4,2,7,2:7,https://boingboing.net/2024/09/18/if-using-ai-...,2024-09-22,en,"If using AI feels like a chore, try this - Boi...",boingboing.net,"Instead, 1minAI bundles everything you need in...",0.998178,...,9,15,company,0.519907,1740a28fad1a24ef3d981c65eb838bafe2f420c5,False,False,1minAI,company,1minai


,canonical_entity,final_type,n_docs,n_rows,n_domains,confidence_mean,keep_for_analysis
0,Openai,company,36066,143536,2581,0.791745,True
1,Chatgpt,technology,33201,118389,2951,0.769613,True
2,Google,company,26450,95393,2436,0.830863,True
3,United States,government_institution,22616,43770,2379,0.758239,True
4,Microsoft,company,22556,74596,2131,0.833017,True


,canonical_entity,final_type,domain,title,date,clean_block_text,confidence,doc_id,block_id,url,clean_block_len
0,#17 Mike’s Famous Philly,government_institution,businesswire.com,SoundHound And Jersey Mike’s Introduce Voice A...,2024-01-24,"Initially live at 50 locations, SoundHound’s v...",0.477525,47114,5,https://www.businesswire.com/news/home/2024012...,351
1,#9 Club Supreme,government_institution,businesswire.com,SoundHound And Jersey Mike’s Introduce Voice A...,2024-01-24,"Initially live at 50 locations, SoundHound’s v...",0.745871,47114,5,https://www.businesswire.com/news/home/2024012...,351
2,$100B+ Life Sciences Market,government_institution,finanznachrichten.de,"Super Micro Computer, Inc.: Supermicro Unveils...",2025-06-11,Delivers high-performing AI infrastructure thr...,0.454479,153982,6,https://www.finanznachrichten.de/nachrichten-2...,171
3,$100B+ Life Sciences Market,government_institution,finanznachrichten.de,KX Launches Its First Agentic AI Blueprint for...,2025-06-11,Delivers high-performing AI infrastructure thr...,0.454228,34586,22,https://www.finanznachrichten.de/nachrichten-2...,171
4,$101m Startup,company,cityam.com,"Musicians know AI is stealing their art, but f...",2023-11-29,Stability AI exec leaves amid concerns over ‘f...,0.634083,12444,38,https://www.cityam.com/musicians-ai-artificial...,86


## Build Sentiment Base Table from Contexts

In [7]:
contexts = entity_contexts_final.copy()
contexts["clean_block_text"] = contexts["clean_block_text"].fillna("").astype(str).map(normalize_spaces)
contexts["title"] = contexts["title"].fillna("").astype(str).map(normalize_spaces)
contexts["domain"] = contexts["domain"].fillna("").astype(str)
contexts["date"] = pd.to_datetime(contexts["date"], errors="coerce")
contexts["confidence"] = pd.to_numeric(contexts["confidence"], errors="coerce")
contexts["canonical_entity"] = contexts["canonical_entity"].fillna("").astype(str).map(normalize_spaces)
contexts["final_type"] = contexts["final_type"].fillna("").astype(str)

contexts = contexts[
    contexts["final_type"].isin(ANALYSIS_KEEP_TYPES)
    & (contexts["clean_block_text"].str.len() >= MIN_CONTEXT_CHARS)
].copy()

contexts["canonical_key"] = contexts["canonical_entity"].map(normalize_entity_key)
contexts["time_bucket"] = build_time_bucket(contexts["date"])
contexts["sample_id"] = (
    contexts["doc_id"].astype(str)
    + "||"
    + contexts["block_id"].astype(str)
    + "||"
    + contexts["canonical_key"].astype(str)
    + "||"
    + contexts["final_type"].astype(str)
)

contexts = contexts.drop_duplicates("sample_id").reset_index(drop=True)

summary_meta = entity_analysis_summary.copy()
summary_meta["canonical_entity"] = summary_meta["canonical_entity"].fillna("").astype(str).map(normalize_spaces)
summary_meta["final_type"] = summary_meta["final_type"].fillna("").astype(str)
summary_meta["canonical_key"] = summary_meta["canonical_entity"].map(normalize_entity_key)

contexts = contexts.merge(
    summary_meta[
        [
            "canonical_entity", "canonical_key", "final_type",
            "n_docs", "n_rows", "n_domains", "confidence_mean"
        ]
    ],
    on=["canonical_entity", "canonical_key", "final_type"],
    how="left",
)

contexts["low_value_entity_flag"] = [
    is_low_value_entity(ent, typ)
    for ent, typ in zip(contexts["canonical_entity"], contexts["final_type"])
]

contexts["entity_doc_bucket"] = np.select(
    [
        contexts["n_docs"] >= HEAD_DOCS_MIN,
        (contexts["n_docs"] >= MID_DOCS_MIN) & (contexts["n_docs"] < HEAD_DOCS_MIN),
        (contexts["n_docs"] >= TAIL_DOCS_MIN) & (contexts["n_docs"] < MID_DOCS_MIN),
    ],
    ["head", "mid", "tail"],
    default="below_tail"
)

contexts["context_text"] = contexts["clean_block_text"].map(lambda x: compact_text(x, MAX_CONTEXT_CHARS))

print("contexts shape after preparation:", contexts.shape)
display(
    contexts[
        [
            "sample_id", "canonical_entity", "final_type", "n_docs", "n_rows",
            "n_domains", "confidence_mean", "entity_doc_bucket", "low_value_entity_flag"
        ]
    ].head(15)
)

contexts shape after preparation: (580671, 21)


,sample_id,canonical_entity,final_type,n_docs,n_rows,n_domains,confidence_mean,entity_doc_bucket,low_value_entity_flag
0,47114||5||#17 mikes famous philly||government_...,#17 Mike’s Famous Philly,government_institution,1,1,1,0.477525,below_tail,False
1,47114||5||#9 club supreme||government_institution,#9 Club Supreme,government_institution,1,1,1,0.745871,below_tail,False
2,153982||6||$100b+ life sciences market||govern...,$100B+ Life Sciences Market,government_institution,2,2,1,0.454353,below_tail,False
3,34586||22||$100b+ life sciences market||govern...,$100B+ Life Sciences Market,government_institution,2,2,1,0.454353,below_tail,False
4,12444||38||$101m startup||company,$101m Startup,company,2,2,1,0.633875,below_tail,False
5,130890||28||$101m startup||company,$101m Startup,company,2,2,1,0.633875,below_tail,False
6,141839||7||$150b valued company||company,$150B Valued Company,company,1,1,1,0.710685,below_tail,False
7,108614||1||$2 855–$2 908||government_institution,$2.855–$2.908,government_institution,2,2,1,0.540364,below_tail,False
8,195028||1||$2 855–$2 908||government_institution,$2.855–$2.908,government_institution,2,2,1,0.540364,below_tail,False
9,15480||0||$3 30–3 70||government_institution,$3.30–3.70,government_institution,1,1,1,0.467257,below_tail,False


## Build Stratified Sentiment Label Pool

In [8]:
def stratified_entity_pool(
    df: pd.DataFrame,
    entity_type: str,
    target_n: int,
) -> pd.DataFrame:
    sub = df[df["final_type"] == entity_type].copy()
    if len(sub) == 0 or target_n <= 0:
        return sub.head(0).copy()

    sub = sub[
        (sub["n_docs"] >= MIN_ENTITY_DOCS.get(entity_type, 0))
        | (sub["n_rows"] >= MIN_ENTITY_ROWS.get(entity_type, 0))
    ].copy()

    sub = sub[~sub["low_value_entity_flag"]].copy()

    head = sub[sub["entity_doc_bucket"] == "head"].copy()
    mid = sub[sub["entity_doc_bucket"] == "mid"].copy()
    tail = sub[sub["entity_doc_bucket"] == "tail"].copy()

    head_n = int(round(target_n * HEAD_SHARE))
    mid_n = int(round(target_n * MID_SHARE))
    tail_n = max(target_n - head_n - mid_n, 0)

    head_sample = sample_with_entity_cap(
        head,
        target_n=head_n,
        entity_col="canonical_entity",
        max_per_entity=MAX_PER_ENTITY_HEAD.get(entity_type, 10),
        seed=SEED,
    )

    mid_sample = sample_with_entity_cap(
        mid,
        target_n=mid_n,
        entity_col="canonical_entity",
        max_per_entity=MAX_PER_ENTITY_MID.get(entity_type, 6),
        seed=SEED,
    )

    tail_sample = sample_with_entity_cap(
        tail,
        target_n=tail_n,
        entity_col="canonical_entity",
        max_per_entity=MAX_PER_ENTITY_TAIL.get(entity_type, 4),
        seed=SEED,
    )

    out = pd.concat([head_sample, mid_sample, tail_sample], ignore_index=True)

    if len(out) < target_n:
        used_ids = set(out["sample_id"].astype(str))
        refill = sub[~sub["sample_id"].isin(used_ids)].copy()
        need = target_n - len(out)
        if len(refill) > 0:
            refill = refill.sample(n=min(need, len(refill)), random_state=SEED)
            out = pd.concat([out, refill], ignore_index=True)

    out = out.drop_duplicates("sample_id").reset_index(drop=True)
    return out

if REUSE_LABEL_POOL_IF_PRESENT and os.path.exists(SENTIMENT_LABEL_POOL_PARQUET):
    label_pool = pd.read_parquet(SENTIMENT_LABEL_POOL_PARQUET)
    print("reused:", SENTIMENT_LABEL_POOL_PARQUET)
else:
    pool_parts = []
    for entity_type, target_n in TARGET_SAMPLES_BY_TYPE.items():
        part = stratified_entity_pool(
            contexts,
            entity_type=entity_type,
            target_n=target_n,
        )
        pool_parts.append(part)

    label_pool = pd.concat(pool_parts, ignore_index=True)
    label_pool = label_pool.drop_duplicates("sample_id").reset_index(drop=True)

    if len(label_pool) > TARGET_TOTAL_SAMPLES:
        label_pool = label_pool.sample(n=TARGET_TOTAL_SAMPLES, random_state=SEED).copy()

    label_pool["sample_source"] = label_pool["final_type"] + "_stratified_context_pool"

    keep_cols = [
        "sample_id",
        "sample_source",
        "doc_id",
        "block_id",
        "date",
        "time_bucket",
        "domain",
        "title",
        "url",
        "canonical_entity",
        "canonical_key",
        "final_type",
        "context_text",
        "confidence",
        "n_docs",
        "n_rows",
        "n_domains",
        "confidence_mean",
        "entity_doc_bucket",
        "low_value_entity_flag",
        "clean_block_len",
    ]
    keep_cols = [c for c in keep_cols if c in label_pool.columns]

    label_pool = label_pool[keep_cols].copy()
    label_pool.to_parquet(SENTIMENT_LABEL_POOL_PARQUET, index=False)
    print("wrote:", SENTIMENT_LABEL_POOL_PARQUET)

print("label_pool shape:", label_pool.shape)
display(label_pool["final_type"].value_counts(dropna=False))
display(label_pool.head(15))

wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_dataset/sentiment_label_pool.parquet
label_pool shape: (9000, 21)


,count
final_type,
company,3200
technology,3200
government_institution,1400
person,1200


,sample_id,sample_source,doc_id,block_id,date,time_bucket,domain,title,url,canonical_entity,...,final_type,context_text,confidence,n_docs,n_rows,n_domains,confidence_mean,entity_doc_bucket,low_value_entity_flag,clean_block_len
0,139594||7||amd||company,company_stratified_context_pool,139594,7,2025-07-15,2025Q3,phoneworld.com.pk,Nvidia and AMD to Restart AI Chips Sales to Ch...,https://www.phoneworld.com.pk/nvidia-and-amd-t...,AMD,...,company,Nvidia and AMD are getting ready to restart sa...,0.992582,2362,8945,518,0.834177,head,False,3013
1,29215||0||amd||company,company_stratified_context_pool,29215,0,2025-08-11,2025Q3,timesofindia.indiatimes.com,"US chip policy: Nvidia, AMD agrees to share 15...",https://timesofindia.indiatimes.com/business/i...,AMD,...,company,"Aug 11, 2025, 19:42 ISTShareAA+Text SizeSmallM...",0.990885,2362,8945,518,0.834177,head,False,2536
2,102868||34||amd||company,company_stratified_context_pool,102868,34,2025-10-28,2025Q4,nasdaq.com,"AMD, U.S. Energy Department Partner On $1 Bln ...",https://www.nasdaq.com/articles/amd-us-energy-...,AMD,...,company,"The Lux AI system, developed by ORNL, AMD, Hew...",0.992993,2362,8945,518,0.834177,head,False,808
3,182714||23||amd||company,company_stratified_context_pool,182714,23,2025-09-04,2025Q3,tomshardware.com,'Doomer science fiction': Nvidia blasts propos...,https://www.tomshardware.com/pc-components/gpu...,AMD,...,company,Nvidia says H20 export controls didn’t stop Ch...,0.989440,2362,8945,518,0.834177,head,False,4496
4,82790||31||amd||company,company_stratified_context_pool,82790,31,2024-03-05,2024Q1,businesstimes.com.sg,AMD hits US roadblock in selling AI chip tailo...,https://www.businesstimes.com.sg/companies-mar...,AMD,...,company,But US officials told AMD that the chip was st...,0.991773,2362,8945,518,0.834177,head,False,353
5,85616||8||amd||company,company_stratified_context_pool,85616,8,2023-09-05,2023Q3,cryptopolitan.com,US Expands AI Chip Export Restrictions to Midd...,https://www.cryptopolitan.com/us-expands-ai-ch...,AMD,...,company,6 Diverse Applications of AI Chips Nvidia and ...,0.990946,2362,8945,518,0.834177,head,False,4784
6,163404||12||amd||company,company_stratified_context_pool,163404,12,2025-08-11,2025Q3,techcentral.co.za,"Nvidia, AMD to give US 15% cut of China AI chi...",https://techcentral.co.za/nvidia-amd-us-15-cut...,AMD,...,company,AMD’s Lisa Su and Nvidia’s Jensen Huang Nvidia...,0.991023,2362,8945,518,0.834177,head,False,3418
7,189888||8||amd||company,company_stratified_context_pool,189888,8,2023-09-05,2023Q3,einpresswire.com,AMD Powers Hitachi Astemo Next-Generation Forw...,https://www.einpresswire.com/article/653880111...,AMD,...,company,"Billions of people, leading Fortune 500 busine...",0.990196,2362,8945,518,0.834177,head,False,321
8,113508||4||aws||company,company_stratified_context_pool,113508,4,2025-11-04,2025Q4,businesswire.com,Cengage Group Teams Up with AWS to Define the ...,https://www.businesswire.com/news/home/2025110...,AWS,...,company,“AWS is proud to expand our collaboration with...,0.985263,1719,4786,513,0.688610,head,False,344
9,189843||0||aws||company,company_stratified_context_pool,189843,0,2025-06-05,2025Q2,hackernoon.com,GISEC Global 2025: Dubai Unites Cyber Defence ...,https://hackernoon.com/gisec-global-2025-dubai...,AWS,...,company,Long; Didn't ReadPeople MentionedCompanies Men...,0.981496,1719,4786,513,0.688610,head,False,2103


## Sentiment Prompt Template

In [9]:
SENTIMENT_SYSTEM_PROMPT = """
Return exactly one JSON object and nothing else.

Task:
Evaluate the AI-related impact sentiment toward the specified entity in the given local context.

Label meaning:
- positive: AI is beneficial, enabling, improving, accelerating, expanding, or strategically favorable for the entity.
- negative: AI is harmful, threatening, restrictive, damaging, costly, displacing, risky, or strategically unfavorable for the entity.
- mixed_or_unclear: impact is mixed, ambiguous, descriptive, neutral, or cannot be assigned confidently.

Rules:
- Judge AI impact on the entity, not general article tone.
- Use only the given local context.
- If context is descriptive without clear directional effect, use mixed_or_unclear.
- Prefer conservative labeling.

Output schema:
{
  "sentiment_label": "positive",
  "confidence": 0.88,
  "reason_short": "Short explanation, <= 24 words."
}
""".strip()

def build_sentiment_prompt(row: pd.Series) -> str:
    payload = {
        "entity": row["canonical_entity"],
        "entity_type": row["final_type"],
        "date": row["date"].isoformat() if pd.notna(row["date"]) else None,
        "domain": row["domain"],
        "title": compact_text(row["title"], 180),
        "context": compact_text(row["context_text"], MAX_CONTEXT_CHARS),
        "entity_stats": {
            "n_docs": int(row["n_docs"]) if pd.notna(row["n_docs"]) else None,
            "n_rows": int(row["n_rows"]) if pd.notna(row["n_rows"]) else None,
            "n_domains": int(row["n_domains"]) if pd.notna(row["n_domains"]) else None,
        },
    }
    return json.dumps(payload, ensure_ascii=False, separators=(",", ":"))

## LLM Labeling Utilities

In [10]:
async def call_deepseek(
    session: aiohttp.ClientSession,
    payload: dict,
    headers: dict,
    sem: asyncio.Semaphore,
) -> dict:
    async with sem:
        for attempt in range(1, LLM_MAX_RETRIES + 1):
            try:
                async with session.post(
                    DEEPSEEK_ENDPOINT,
                    headers=headers,
                    json=payload,
                    timeout=LLM_TIMEOUT_S,
                ) as resp:
                    txt = await resp.text()

                    if resp.status in (429, 500, 502, 503, 504):
                        await asyncio.sleep(min(2 ** attempt, 20))
                        continue

                    if resp.status != 200:
                        return {"_error": f"http_{resp.status}", "_raw": txt[:500]}

                    data = json.loads(txt)
                    content = data["choices"][0]["message"]["content"]

                    if not content or not content.strip():
                        await asyncio.sleep(min(2 ** attempt, 20))
                        continue

                    return {"_ok": True, "content": content}

            except Exception as e:
                if attempt == LLM_MAX_RETRIES:
                    return {"_error": "exception", "_raw": str(e)[:500]}
                await asyncio.sleep(min(2 ** attempt, 20))

        return {"_error": "max_retries"}

async def label_sentiment_pool(df: pd.DataFrame, out_jsonl: str, id_col: str):
    done = load_done_ids(out_jsonl, id_col=id_col)
    todo = df[~df[id_col].isin(done)].copy()

    print("=" * 100)
    print("Sentiment labeling job summary")
    print(f"total rows in pool   : {len(df):,}")
    print(f"already labeled      : {len(done):,}")
    print(f"remaining this run   : {len(todo):,}")
    print(f"output jsonl         : {out_jsonl}")
    print(f"model                : {DEEPSEEK_MODEL}")
    print(f"concurrency          : {LLM_CONCURRENCY}")
    print("=" * 100)

    if len(todo) == 0:
        print("No remaining rows. Skipping sentiment labeling.")
        return

    headers = {
        "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
        "Content-Type": "application/json",
    }

    sem = asyncio.Semaphore(LLM_CONCURRENCY)
    connector = aiohttp.TCPConnector(limit=LLM_CONCURRENCY)
    timeout = aiohttp.ClientTimeout(total=LLM_TIMEOUT_S)

    started_at = time.time()
    completed = 0
    status_counter = {"ok": 0, "parse_error": 0, "error": 0}
    flush_buffer = []

    async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
        tasks = []

        for _, row in todo.iterrows():
            payload = {
                "model": DEEPSEEK_MODEL,
                "messages": [
                    {"role": "system", "content": SENTIMENT_SYSTEM_PROMPT},
                    {"role": "user", "content": build_sentiment_prompt(row)},
                ],
                "temperature": 0,
                "max_tokens": 160,
                "response_format": {"type": "json_object"},
            }
            meta = to_jsonable(row.to_dict())
            tasks.append((row[id_col], payload, meta))

        async def runner(sample_id, payload, meta):
            res = await call_deepseek(session, payload, headers, sem)
            out = {id_col: to_jsonable(sample_id), **meta}

            if "_ok" in res:
                out["_raw_json"] = res["content"]
                try:
                    parsed = json.loads(res["content"])
                    out.update(to_jsonable(parsed))
                    out["label_status"] = "ok"
                except Exception:
                    out["label_status"] = "parse_error"
            else:
                out["label_status"] = "error"
                out["error"] = res.get("_error", "unknown")
                out["error_raw"] = res.get("_raw", "")[:300]

            return out

        coros = [runner(sample_id, payload, meta) for sample_id, payload, meta in tasks]
        iterator = tqdm(asyncio.as_completed(coros), total=len(coros), desc="Sentiment labeling")

        with open(out_jsonl, "a", encoding="utf-8") as f:
            for fut in iterator:
                out = await fut
                flush_buffer.append(json.dumps(to_jsonable(out), ensure_ascii=False))

                completed += 1
                label_status = out.get("label_status", "error")
                if label_status not in status_counter:
                    status_counter[label_status] = 0
                status_counter[label_status] += 1

                if len(flush_buffer) >= JSONL_FLUSH_EVERY:
                    f.write("\n".join(flush_buffer) + "\n")
                    f.flush()
                    flush_buffer = []

                if completed % CHECKPOINT_EVERY == 0 or completed == len(todo):
                    elapsed = time.time() - started_at
                    rate = completed / elapsed if elapsed > 0 else 0.0
                    print(
                        f"[checkpoint] completed={completed:,}/{len(todo):,} | "
                        f"elapsed_min={elapsed/60:.2f} | rate_per_sec={rate:.2f} | "
                        f"ok={status_counter.get('ok',0):,} | "
                        f"parse_error={status_counter.get('parse_error',0):,} | "
                        f"error={status_counter.get('error',0):,}"
                    )

            if flush_buffer:
                f.write("\n".join(flush_buffer) + "\n")
                f.flush()

    elapsed = time.time() - started_at
    print("=" * 100)
    print("Sentiment labeling finished")
    print(f"completed            : {completed:,}")
    print(f"elapsed_minutes      : {elapsed/60:.2f}")
    print(f"status_counts        : {status_counter}")
    print("=" * 100)

## Run or Reuse LLM Sentiment Labels

In [11]:
need_rebuild_llm_labels = True

if REUSE_LLM_LABELS_IF_PRESENT and os.path.exists(SENTIMENT_LLM_LABELS_PARQUET):
    try:
        sentiment_labels = pd.read_parquet(SENTIMENT_LLM_LABELS_PARQUET)
        required_cols = {"sample_id", "sentiment_label", "confidence", "label_status"}
        if required_cols.issubset(sentiment_labels.columns):
            need_rebuild_llm_labels = False
            print("reused:", SENTIMENT_LLM_LABELS_PARQUET)
        else:
            print("existing sentiment labels schema is incompatible; rebuilding.")
    except Exception as e:
        print("failed to load existing sentiment labels; rebuilding.", e)

if need_rebuild_llm_labels:
    assert USE_LLM_LABELING, "Missing DEEPSEEK_API_KEY for LLM sentiment labeling."

    if os.path.exists(SENTIMENT_LLM_LABELS_JSONL):
        print("Existing JSONL found. Resuming with done-id tracking:")
        print(SENTIMENT_LLM_LABELS_JSONL)

    try:
        _loop = asyncio.get_running_loop()
        _has_running_loop = _loop.is_running()
    except RuntimeError:
        _loop = None
        _has_running_loop = False

    if _has_running_loop:
        await label_sentiment_pool(
            label_pool,
            SENTIMENT_LLM_LABELS_JSONL,
            id_col="sample_id",
        )
    else:
        asyncio.run(
            label_sentiment_pool(
                label_pool,
                SENTIMENT_LLM_LABELS_JSONL,
                id_col="sample_id",
            )
        )

    sentiment_labels = pd.read_json(SENTIMENT_LLM_LABELS_JSONL, lines=True)
    sentiment_labels.to_parquet(SENTIMENT_LLM_LABELS_PARQUET, index=False)
    print("wrote:", SENTIMENT_LLM_LABELS_PARQUET)

print("sentiment_labels shape:", sentiment_labels.shape)
display(sentiment_labels["label_status"].value_counts(dropna=False))
display(sentiment_labels.head(10))

Sentiment labeling job summary
total rows in pool   : 9,000
already labeled      : 0
remaining this run   : 9,000
output jsonl         : /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_dataset/sentiment_llm_labels.jsonl
model                : deepseek-chat
concurrency          : 48


Sentiment labeling:   0%|          | 0/9000 [00:00<?, ?it/s]

[checkpoint] completed=200/9,000 | elapsed_min=0.21 | rate_per_sec=15.93 | ok=200 | parse_error=0 | error=0
[checkpoint] completed=400/9,000 | elapsed_min=0.34 | rate_per_sec=19.46 | ok=400 | parse_error=0 | error=0
[checkpoint] completed=600/9,000 | elapsed_min=0.47 | rate_per_sec=21.12 | ok=600 | parse_error=0 | error=0
[checkpoint] completed=800/9,000 | elapsed_min=0.61 | rate_per_sec=21.86 | ok=800 | parse_error=0 | error=0
[checkpoint] completed=1,000/9,000 | elapsed_min=0.75 | rate_per_sec=22.25 | ok=1,000 | parse_error=0 | error=0
[checkpoint] completed=1,200/9,000 | elapsed_min=0.88 | rate_per_sec=22.64 | ok=1,200 | parse_error=0 | error=0
[checkpoint] completed=1,400/9,000 | elapsed_min=1.02 | rate_per_sec=22.85 | ok=1,400 | parse_error=0 | error=0
[checkpoint] completed=1,600/9,000 | elapsed_min=1.16 | rate_per_sec=23.07 | ok=1,600 | parse_error=0 | error=0
[checkpoint] completed=1,800/9,000 | elapsed_min=1.29 | rate_per_sec=23.30 | ok=1,800 | parse_error=0 | error=0
[checkpo

,count
label_status,
ok,8999
error,1


,sample_id,sample_source,doc_id,block_id,date,time_bucket,domain,title,url,canonical_entity,...,confidence_mean,entity_doc_bucket,low_value_entity_flag,clean_block_len,_raw_json,sentiment_label,reason_short,label_status,error,error_raw
0,93375||8||jordan||government_institution,government_institution_stratified_context_pool,93375,8,2023-06-05,2023Q2,telecom.economictimes.indiatimes.com,OpenAI CEO sees 'huge' Israeli role in reducin...,https://telecom.economictimes.indiatimes.com/a...,Jordan,...,0.797979,mid,False,226,"{\n ""sentiment_label"": ""mixed_or_unclear"",\n ...",mixed_or_unclear,Context describes travel plans without specify...,ok,NaN,NaN
1,94750||9||quantum computing||technology,technology_stratified_context_pool,94750,9,2022-06-07,2022Q2,newsbytes.ph,"After AI and blockchain, IBM tags quantum comp...",https://newsbytes.ph/2022/06/06/after-ai-and-b...,Quantum Computing,...,0.873885,head,False,936,"{\n ""sentiment_label"": ""positive"",\n ""confid...",positive,IBM emphasizes quantum computing's exponential...,ok,NaN,NaN
2,23320||18||quantum computing||technology,technology_stratified_context_pool,23320,18,2025-02-10,2025Q1,einpresswire.com,LEAP 2025 Opens with Announcement of Record-br...,https://www.einpresswire.com/article/784621089...,Quantum Computing,...,0.873885,head,False,487,"{\n ""sentiment_label"": ""positive"",\n ""confid...",positive,Quantum computing described as breakthrough en...,ok,NaN,NaN
3,134159||23||dell||company,company_stratified_context_pool,134159,23,2024-05-28,2024Q2,computerweekly.com,Executive Interview: Why Dell wants to be your...,https://www.computerweekly.com/news/366586034/...,Dell,...,0.850091,head,False,171,"{\n ""sentiment_label"": ""positive"",\n ""confid...",positive,"Dell positions AI as central to its strategy, ...",ok,NaN,NaN
4,190378||7||quantum computing||technology,technology_stratified_context_pool,190378,7,2024-11-04,2024Q4,businessday.ng,Adopting quantum computing and artificial inte...,https://businessday.ng/opinion/article/adoptin...,Quantum Computing,...,0.873885,head,False,2313,"{\n ""sentiment_label"": ""positive"",\n ""confid...",positive,"Quantum computing enhances healthcare, agricul...",ok,NaN,NaN
5,178832||10||dell||company,company_stratified_context_pool,178832,10,2024-05-25,2024Q2,techtarget.com,Dell AI Factory curates AI tech for customers ...,https://www.techtarget.com/searchdatacenter/ne...,Dell,...,0.850091,head,False,438,"{\n ""sentiment_label"": ""positive"",\n ""confid...",positive,"Dell positions AI as beneficial, offering inte...",ok,NaN,NaN
6,164822||12||intelligent agents||technology,technology_stratified_context_pool,164822,12,2025-08-28,2025Q3,mexc.com,"""Bazaar"" surpasses ""Cathedral"", how does crypt...",https://www.mexc.com/fi-FI/news/1093,Intelligent Agents,...,0.841947,tail,False,552,"{\n ""sentiment_label"": ""positive"",\n ""confid...",positive,AI agents expand task capabilities and enable ...,ok,NaN,NaN
7,93712||1||signalfire||company,company_stratified_context_pool,93712,1,2023-09-26,2023Q3,wlbt.com,Kolena Secures Series A Investment Led by Lobb...,https://www.wlbt.com/prnewswire/2023/09/26/kol...,Signalfire,...,0.601631,tail,False,5471,"{\n ""sentiment_label"": ""positive"",\n ""confid...",positive,"AI testing enables safe, reliable AI systems, ...",ok,NaN,NaN
8,104550||3||federal communications commission||...,government_institution_stratified_context_pool,104550,3,2024-07-24,2024Q3,broadbandbreakfast.com,Chamber of Commerce Calls FCC's AI Proposal 'P...,https://broadbandbreakfast.com/chamber-of-comm...,Federal Communications Commission,...,0.706639,mid,False,2415,"{\n ""sentiment_label"": ""negative"",\n ""co...",negative,Chamber criticizes FCC's AI proposal as premat...,ok,NaN,NaN
9,77390||4||dell||company,company_stratified_context_pool,77390,4,2023-05-31,2023Q2,forbes.com,"Dell TechWorld 2023 — Security, AI And Multicloud",https://www.forbes.com/sites/moorinsights/2023...,Dell,...,0.850091,head,False,6204,"{\n ""sentiment_label"": "

## Build Final Training Dataset

In [12]:
sentiment_ok = sentiment_labels[sentiment_labels["label_status"] == "ok"].copy()
sentiment_ok = sentiment_ok[sentiment_ok["sentiment_label"].isin(VALID_SENTIMENT_LABELS)].copy()

sentiment_ok["confidence"] = pd.to_numeric(sentiment_ok["confidence"], errors="coerce")
sentiment_ok["confidence"] = sentiment_ok["confidence"].clip(lower=0.0, upper=1.0)

sentiment_ok = sentiment_ok[sentiment_ok["context_text"].fillna("").str.len() >= MIN_CONTEXT_CHARS].copy()
sentiment_ok = sentiment_ok.drop_duplicates("sample_id").reset_index(drop=True)

training_keep_cols = [
    "sample_id",
    "sample_source",
    "doc_id",
    "block_id",
    "date",
    "time_bucket",
    "domain",
    "title",
    "url",
    "canonical_entity",
    "canonical_key",
    "final_type",
    "context_text",
    "n_docs",
    "n_rows",
    "n_domains",
    "confidence_mean",
    "entity_doc_bucket",
    "sentiment_label",
    "confidence",
    "reason_short",
]
training_keep_cols = [c for c in training_keep_cols if c in sentiment_ok.columns]

training_data = sentiment_ok[training_keep_cols].copy()
training_data.to_parquet(SENTIMENT_TRAINING_DATA_PARQUET, index=False)

print("wrote:", SENTIMENT_TRAINING_DATA_PARQUET)
print("training_data shape:", training_data.shape)
display(training_data.head(15))

wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_dataset/sentiment_training_data.parquet
training_data shape: (8999, 21)


,sample_id,sample_source,doc_id,block_id,date,time_bucket,domain,title,url,canonical_entity,...,final_type,context_text,n_docs,n_rows,n_domains,confidence_mean,entity_doc_bucket,sentiment_label,confidence,reason_short
0,93375||8||jordan||government_institution,government_institution_stratified_context_pool,93375,8,2023-06-05,2023Q2,telecom.economictimes.indiatimes.com,OpenAI CEO sees 'huge' Israeli role in reducin...,https://telecom.economictimes.indiatimes.com/a...,Jordan,...,government_institution,After crisscrossing Europe last month meeting ...,90,143,70,0.797979,mid,mixed_or_unclear,0.65,Context describes travel plans without specify...
1,94750||9||quantum computing||technology,technology_stratified_context_pool,94750,9,2022-06-07,2022Q2,newsbytes.ph,"After AI and blockchain, IBM tags quantum comp...",https://newsbytes.ph/2022/06/06/after-ai-and-b...,Quantum Computing,...,technology,IBM’s director of Quantum Infrastructure Jerry...,658,1315,291,0.873885,head,positive,0.85,IBM emphasizes quantum computing's exponential...
2,23320||18||quantum computing||technology,technology_stratified_context_pool,23320,18,2025-02-10,2025Q1,einpresswire.com,LEAP 2025 Opens with Announcement of Record-br...,https://www.einpresswire.com/article/784621089...,Quantum Computing,...,technology,“A breakthrough I think is only about three-to...,658,1315,291,0.873885,head,positive,0.85,Quantum computing described as breakthrough en...
3,134159||23||dell||company,company_stratified_context_pool,134159,23,2024-05-28,2024Q2,computerweekly.com,Executive Interview: Why Dell wants to be your...,https://www.computerweekly.com/news/366586034/...,Dell,...,company,"At Dell Technologies World in Las Vegas, artif...",670,2256,278,0.850091,head,positive,0.85,"Dell positions AI as central to its strategy, ..."
4,190378||7||quantum computing||technology,technology_stratified_context_pool,190378,7,2024-11-04,2024Q4,businessday.ng,Adopting quantum computing and artificial inte...,https://businessday.ng/opinion/article/adoptin...,Quantum Computing,...,technology,Healthcare: Quantum computing can enhance drug...,658,1315,291,0.873885,head,positive,0.85,"Quantum computing enhances healthcare, agricul..."
5,178832||10||dell||company,company_stratified_context_pool,178832,10,2024-05-25,2024Q2,techtarget.com,Dell AI Factory curates AI tech for customers ...,https://www.techtarget.com/searchdatacenter/ne...,Dell,...,company,LAS VEGAS -- Dell made it clear at its annual ...,670,2256,278,0.850091,head,positive,0.85,"Dell positions AI as beneficial, offering inte..."
6,164822||12||intelligent agents||technology,technology_stratified_context_pool,164822,12,2025-08-28,2025Q3,mexc.com,"""Bazaar"" surpasses ""Cathedral"", how does crypt...",https://www.mexc.com/fi-FI/news/1093,Intelligent Agents,...,technology,"Currently, humans undertake most of the servic...",46,94,26,0.841947,tail,positive,0.85,AI agents expand task capabilities and enable ...
7,93712||1||signalfire||company,company_stratified_context_pool,93712,1,2023-09-26,2023Q3,wlbt.com,Kolena Secures Series A Investment Led by Lobb...,https://www.wlbt.com/prnewswire/2023/09/26/kol...,Signalfire,...,company,ReleasesPromote Your BusinessKolena Secures Se...,22,23,22,0.601631,tail,positive,0.85,"AI testing enables safe, reliable AI systems, ..."
8,104550||3||federal communications commission||...,government_institution_stratified_context_pool,104550,3,2024-07-24,2024Q3,broadbandbreakfast.com,Chamber of Commerce Calls FCC's AI Proposal 'P...,https://broadbandbreakfast.com/chamber-of-comm...,Federal Communications Commission,...,government_institution,"Jul 24, 2024 — 1 min read Photo of Jordan Cren...",158,170,124,0.706639,mid,negative,0.75,Chamber criticizes FCC's AI proposal as premat...
9,77390||4||dell||company,company_stratified_context_pool,77390,4,2023-05-31,2023Q2,forbes.com,"Dell TechWorld 2023 — Security, AI And Multicloud",https://www.forbes.com/sites/moorinsights/2023...,Dell,...,company,"At DTW, Dell took it one

## Summary Tables and Report

In [13]:
label_summary = (
    training_data.groupby(["final_type", "sentiment_label"], dropna=False)
    .agg(
        n_samples=("sample_id", "size"),
        n_entities=("canonical_entity", "nunique"),
        n_docs=("doc_id", "nunique"),
        confidence_mean=("confidence", "mean"),
    )
    .reset_index()
    .sort_values(["final_type", "n_samples"], ascending=[True, False])
)

label_summary.to_csv(SENTIMENT_LABEL_SUMMARY_CSV, index=False)
print("wrote:", SENTIMENT_LABEL_SUMMARY_CSV)

report = {
    "label_pool_rows": int(len(label_pool)),
    "llm_label_rows_total": int(len(sentiment_labels)),
    "llm_label_rows_ok": int(len(sentiment_ok)),
    "training_rows": int(len(training_data)),
    "training_distribution": label_summary.to_dict(orient="records"),
    "training_rows_by_type": training_data["final_type"].value_counts(dropna=False).to_dict(),
    "sentiment_distribution": training_data["sentiment_label"].value_counts(dropna=False).to_dict(),
}

with open(SENTIMENT_LABELING_REPORT_JSON, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print("wrote:", SENTIMENT_LABELING_REPORT_JSON)
print(report)

wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_dataset/sentiment_label_summary.csv
wrote: /content/drive/MyDrive/NLP_FINAL_PROJECT_Tom_Chen/output/05_sentiment_dataset/sentiment_labeling_report.json
{'label_pool_rows': 9000, 'llm_label_rows_total': 9000, 'llm_label_rows_ok': 8999, 'training_rows': 8999, 'training_distribution': [{'final_type': 'company', 'sentiment_label': 'positive', 'n_samples': 2150, 'n_entities': 1213, 'n_docs': 2013, 'confidence_mean': 0.8676325581395349}, {'final_type': 'company', 'sentiment_label': 'mixed_or_unclear', 'n_samples': 726, 'n_entities': 434, 'n_docs': 638, 'confidence_mean': 0.7393250688705234}, {'final_type': 'company', 'sentiment_label': 'negative', 'n_samples': 324, 'n_entities': 178, 'n_docs': 299, 'confidence_mean': 0.8359567901234567}, {'final_type': 'government_institution', 'sentiment_label': 'positive', 'n_samples': 576, 'n_entities': 251, 'n_docs': 554, 'confidence_mean': 0.8594618055555556}, {'final_type': 'g

## Quick Readout

In [14]:
label_pool_preview = pd.read_parquet(SENTIMENT_LABEL_POOL_PARQUET)
sentiment_labels_preview = pd.read_parquet(SENTIMENT_LLM_LABELS_PARQUET)
training_data_preview = pd.read_parquet(SENTIMENT_TRAINING_DATA_PARQUET)

print("label_pool shape:", label_pool_preview.shape)
print("sentiment_labels shape:", sentiment_labels_preview.shape)
print("training_data shape:", training_data_preview.shape)

print("\nlabel pool by entity type:")
display(label_pool_preview["final_type"].value_counts(dropna=False))

print("\nlabel status distribution:")
display(sentiment_labels_preview["label_status"].value_counts(dropna=False))

print("\ntraining sentiment distribution:")
display(training_data_preview["sentiment_label"].value_counts(dropna=False))

print("\ntraining rows by entity type:")
display(training_data_preview["final_type"].value_counts(dropna=False))

print("\ntraining confidence summary:")
display(
    training_data_preview["confidence"].describe(
        percentiles=[0.01, 0.1, 0.5, 0.9, 0.99]
    )
)

print("\nsummary table:")
display(label_summary)

print("\nexample labeled rows:")
display(
    training_data_preview[
        [
            "canonical_entity",
            "final_type",
            "sentiment_label",
            "confidence",
            "domain",
            "title",
            "context_text",
        ]
    ].head(20)
)

label_pool shape: (9000, 21)
sentiment_labels shape: (9000, 27)
training_data shape: (8999, 21)

label pool by entity type:


,count
final_type,
company,3200
technology,3200
government_institution,1400
person,1200



label status distribution:


,count
label_status,
ok,8999
error,1



training sentiment distribution:


...

,confidence
count,8999.000000
mean,0.825986
std,0.060712
min,0.500000
1%,0.650000
10%,0.750000
50%,0.850000
90%,0.920000
99%,0.920000
max,0.950000



summary table:


,final_type,sentiment_label,n_samples,n_entities,n_docs,confidence_mean
2,company,positive,2150,1213,2013,0.867633
0,company,mixed_or_unclear,726,434,638,0.739325
1,company,negative,324,178,299,0.835957
5,government_institution,positive,576,251,554,0.859462
3,government_institution,mixed_or_unclear,528,270,496,0.746117
4,government_institution,negative,296,153,272,0.840912
6,person,mixed_or_unclear,488,302,470,0.746209
8,person,positive,464,322,460,0.855776
7,person,negative,247,152,238,0.844089
11,technology,positive,1984,1019,1819,0.859713



example labeled rows:


,canonical_entity,final_type,sentiment_label,confidence,domain,title,context_text
0,Jordan,government_institution,mixed_or_unclear,0.65,telecom.economictimes.indiatimes.com,OpenAI CEO sees 'huge' Israeli role in reducin...,After crisscrossing Europe last month meeting ...
1,Quantum Computing,technology,positive,0.85,newsbytes.ph,"After AI and blockchain, IBM tags quantum comp...",IBM’s director of Quantum Infrastructure Jerry...
2,Quantum Computing,technology,positive,0.85,einpresswire.com,LEAP 2025 Opens with Announcement of Record-br...,“A breakthrough I think is only about three-to...
3,Dell,company,positive,0.85,computerweekly.com,Executive Interview: Why Dell wants to be your...,"At Dell Technologies World in Las Vegas, artif..."
4,Quantum Computing,technology,positive,0.85,businessday.ng,Adopting quantum computing and artificial inte...,Healthcare: Quantum computing can enhance drug...
5,Dell,company,positive,0.85,techtarget.com,Dell AI Factory curates AI tech for customers ...,LAS VEGAS -- Dell made it clear at its annual ...
6,Intelligent Agents,technology,positive,0.85,mexc.com,"""Bazaar"" surpasses ""Cathedral"", how does crypt...","Currently, humans undertake most of the servic..."
7,Signalfire,company,positive,0.85,wlbt.com,Kolena Secures Series A Investment Led by Lobb...,ReleasesPromote Your BusinessKolena Secures Se...
8,Federal Communications Commission,government_institution,negative,0.75,broadbandbreakfast.com,Chamber of Commerce Calls FCC's AI Proposal 'P...,"Jul 24, 2024 — 1 min read Photo of Jordan Cren..."
9,Dell,company,positive,0.85,forbes.com,"Dell TechWorld 2023 — Security, AI And Multicloud","At DTW, Dell took it one step further by annou..."
